# Executors

In [ ]:
from agent_framework import (
    WorkflowContext,
    executor,
    WorkflowBuilder,
)

@executor(id="upper_case_executor")
async def upper_case(text: str, ctx: WorkflowContext[str, str]) -> None:
    """Convert the input to uppercase and forward it to the next node."""
    await ctx.send_message(text.upper())


@executor(id="reverse_executor")
async def reverse(text: str, ctx: WorkflowContext[str, str]) -> None:
    """Reverse the input text and forward it to the next node."""
    await ctx.send_message(text[::-1])

In [14]:
workflow = WorkflowBuilder(
    start_executor=upper_case,
    output_from=[reverse],
    intermediate_output_from=[upper_case],
).add_edge(upper_case, reverse).build()

In [ ]:
# Streaming (events are not collected in a list, they are processed as they occur)
async for event in workflow.run('Hello World', stream=True):
    if event.type == "executor_invoked":
        print(f"Starting {event.executor_id}")
    elif event.type == "executor_completed":
        print(f"Completed {event.executor_id}: {event.data}")
    elif event.type == "intermediate":
        print(f"Intermediate output from {event.executor_id}: {event.data}")
    elif event.type == "output":
        print(f"Terminal output: {event.data}")
    elif event.type == "error":
        print(f"Workflow error: {event.data}")

Starting upper_case_executor
Completed upper_case_executor: ['HELLO WORLD']
Starting reverse_executor
Completed reverse_executor: ['DLROW OLLEH']


In [21]:
# Batch (events are collected in a list)
events = await workflow.run('Hello World', stream=False)
print([event for event in events if event.type == "executor_completed"])
# Note: the executors didn't provide outputs, so the output lists will be empty
print(events.get_outputs())
print(events.get_intermediate_outputs())

[WorkflowEvent(type='executor_completed', executor_id='upper_case_executor', data=['HELLO WORLD']), WorkflowEvent(type='executor_completed', executor_id='reverse_executor', data=['DLROW OLLEH'])]
[]
[]


## Terminal and Intermediate Output Executors

`ctx.yield_output(...)` has no per-emission flag. The same call is labelled "output" or "intermediate" solely based on the builder's designation. There is no `ctx.yield_intermediate(...)` API — designation does not vary per yield.

Both lists are optional. If either output-selection list is provided, an executor that appears in neither list can still send messages to downstream executors via ctx.send_message(...), but its yield_output calls are hidden. If both lists are omitted, every yield_output still emits "output" for compatibility.

In [ ]:
from agent_framework import WorkflowBuilder

workflow = WoarkflowBuilder(
    start_executor=analysis_executor,
    output_from=[summary_executor],
    intermediate_output_from=[analysis_executor],
).build()

## Edges

| Type                      | Description                                             | Use case                 |
|---------------------------|---------------------------------------------------------|--------------------------|
| Direct                    | Simple one-to-one connections                           | Linear pipelines         |
| Conditional               | Edges with conditions that determine when messages flow | Binary routing (if/else) |
| Switch-Case               | Route to different executors based on conditions        | Multi-branch routing     |
| Multi-Selection (Fan-out) | One executor sending messages to multiple targets       | Parallel processing      |
| Fan-in                    | Multiple executors sending to a single target           | Aggregation              |


In [34]:
@executor(id="add_padding")
async def add_padding(text: str, ctx: WorkflowContext[str, None]) -> None:
    """Add padding to the input text and forward it to the next node."""
    await ctx.send_message(f"  {text}  ")

@executor(id="aggregator")
async def aggregator(text: list[str], ctx: WorkflowContext[str, None]) -> None:
    """Aggregate the input text and forward it to the next node."""
    # Receives a list of input types
    print(type(text), len(text), [type(item) for item in text], text)
    await ctx.send_message(f"[{text}]")

#### Direct Edges (linear, 1-to-1)

In [22]:
workflow = WorkflowBuilder(
    start_executor=upper_case,
    output_from=[reverse],
    intermediate_output_from=[upper_case],
).add_edge(upper_case, reverse).build()
events = await workflow.run('Hello World', stream=False)
print([event for event in events if event.type == "executor_completed"])

[WorkflowEvent(type='executor_completed', executor_id='upper_case_executor', data=['HELLO WORLD']), WorkflowEvent(type='executor_completed', executor_id='reverse_executor', data=['DLROW OLLEH'])]


In [35]:
# Here we use chaining to simplify
workflow = WorkflowBuilder(
    start_executor=upper_case,
).add_chain([upper_case, reverse, add_padding]).build()

async for event in workflow.run('Hello World', stream=True):
    if event.type == "executor_invoked":
        print(f"Starting {event.executor_id}")
    elif event.type == "executor_completed":
        print(f"Completed {event.executor_id}: {event.data}")
    elif event.type == "intermediate":
        print(f"Intermediate output from {event.executor_id}: {event.data}")
    elif event.type == "output":
        print(f"Terminal output: {event.data}")
    elif event.type == "error":
        print(f"Workflow error: {event.data}")

Starting upper_case_executor
Completed upper_case_executor: ['HELLO WORLD']
Starting reverse_executor
Completed reverse_executor: ['DLROW OLLEH']
Starting add_padding
Completed add_padding: ['  DLROW OLLEH  ']


#### Fan-In Edges

Collect messages from multiple sources into a single target

#### Fan-Out Edges

Same as adding individual `add_edge`. The only benefit to adding edges independently is you can add conditions to them.

In [33]:
builder = WorkflowBuilder(start_executor=upper_case)
builder.add_fan_out_edges(upper_case, [reverse, add_padding])
builder.add_fan_in_edges([reverse, add_padding], aggregator)
workflow = builder.build()

async for event in workflow.run('Hello World', stream=True):
    if event.type == "executor_invoked":
        print(f"Starting {event.executor_id}")
    elif event.type == "executor_completed":
        print(f"Completed {event.executor_id}: {event.data}")
    elif event.type == "intermediate":
        print(f"Intermediate output from {event.executor_id}: {event.data}")
    elif event.type == "output":
        print(f"Terminal output: {event.data}")
    elif event.type == "error":
        print(f"Workflow error: {event.data}")

Starting upper_case_executor
Completed upper_case_executor: ['HELLO WORLD']
Starting reverse_executor
Completed reverse_executor: ['DLROW OLLEH']
Starting add_padding
Completed add_padding: ['  HELLO WORLD  ']
<class 'list'> 2 [<class 'str'>, <class 'str'>] ['DLROW OLLEH', '  HELLO WORLD  ']
Starting aggregator
Completed aggregator: ["[['DLROW OLLEH', '  HELLO WORLD  ']]"]


#### Conditional Edges

Conditional edges allow your workflow to make routing decisions based on the content or properties of messages flowing through the workflow.

In [38]:
builder = WorkflowBuilder(start_executor=upper_case)
# Condition is evaluated on the output of upper_case to determine if the edge should run
builder.add_edge(upper_case, reverse, condition=lambda text: text != "HELLO WORLD")
builder.add_edge(reverse, add_padding)
# Fall through if the condition for the reverse edge is not met
builder.add_edge(upper_case, add_padding)
workflow = builder.build()

async for event in workflow.run('Hello World', stream=True):
    if event.type == "executor_invoked":
        print(f"Starting {event.executor_id}")
    elif event.type == "executor_completed":
        print(f"Completed {event.executor_id}: {event.data}")
    elif event.type == "intermediate":
        print(f"Intermediate output from {event.executor_id}: {event.data}")
    elif event.type == "output":
        print(f"Terminal output: {event.data}")
    elif event.type == "error":
        print(f"Workflow error: {event.data}")

Starting upper_case_executor
Completed upper_case_executor: ['HELLO WORLD']
Starting add_padding
Completed add_padding: ['  HELLO WORLD  ']
